## 1. Чтение, форматироание

In [27]:
import pandas as pd
import numpy as np
import requests
pd.options.display.float_format = '{:.2f}'.format

In [28]:
df = pd.read_json("../ex02/auto.json")
df

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus
...,...,...,...,...,...
720,Y163O8161RUS,2,1600.00,Ford,Focus
721,M0309X197RUS,1,22300.00,Ford,Focus
722,O673E8197RUS,2,600.00,Ford,Focus
723,8610T8154RUS,1,2000.00,Ford,Focus


## 2. Расширяем датасет

- с помощью sample и задаем сид генерации 21

In [29]:
ex_df = df.sample(n=200, random_state=21)
ex_df["Refund"] = df["Refund"].sample(n=200, random_state=21).values
ex_df["Fines"] = df["Fines"].sample(n=200, random_state=21).values
concat_rows = pd.concat([df, ex_df], ignore_index=True)
concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        925
dtype: int64

## 3. Добавляем новый столбец года

In [30]:
np.random.seed(21)

- рандомная генерация значений через нампи

In [31]:
years = np.random.randint(1980, 2020, size=len(concat_rows))

fines = pd.concat([concat_rows, pd.Series(years, name = "Year")], axis = 1)
fines.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        925
Year         925
dtype: int64

## 4. Добавляем фамилии

- Запрос был на series, так что я забрал только 1 столбец из прочианного json

In [32]:
# !pwd
raw_surs = pd.read_json("../../datasets/surname.json")
print(raw_surs)
s_surs = raw_surs[0]
type(s_surs)

            0        1     2
0        NAME    COUNT  RANK
1       ADAMS   427865    42
2       ALLEN   482607    33
3     ALVAREZ   233983    92
4    ANDERSON   784404    15
..        ...      ...   ...
96   WILLIAMS  1625252     3
97     WILSON   801882    14
98       WOOD   250715    84
99     WRIGHT   458980    35
100     YOUNG   484447    32

[101 rows x 3 columns]


pandas.core.series.Series

- считаю сколько уникальных номеров

In [33]:
uniq_num = fines.nunique()
uniq_num


CarNumber    531
Refund         2
Fines        192
Make           7
Model         10
Year          40
dtype: int64

- дублируем фамилии под кол-во уникальных номеров

In [34]:
surnames = s_surs.sample(n=uniq_num["CarNumber"], replace=True, random_state=21)
surnames = surnames.reset_index(drop=True)

- создаем фрейм

In [35]:
owners = pd.DataFrame(
    {
    "CarNumber": fines["CarNumber"].unique(),
    "SURNAME": surnames
})
owners

,CarNumber,SURNAME
0,Y163O8161RUS,REYES
1,E432XX77RUS,ROGERS
2,7184TT36RUS,MORALES
3,X582HE161RUS,ANDERSON
4,92918M178RUS,LONG
...,...,...
526,O136HO197RUS,REED
527,O22097197RUS,MILLER
528,M0309X197RUS,TURNER
529,O673E8197RUS,BROOKS


- создаем 5 новых значений и подпихиваем в fines

In [36]:
fines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  925 non-null    object 
 1   Refund     925 non-null    int64  
 2   Fines      925 non-null    float64
 3   Make       925 non-null    object 
 4   Model      925 non-null    object 
 5   Year       925 non-null    int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 43.5+ KB


In [37]:
temp = pd.DataFrame([
    {"CarNumber": "A111AA77RUS", "Refund": 0, "Fines": 1800.0, "Make": "Toyota", "Model": "Camry", "Year": 2007},
    {"CarNumber": "B222BB99RUS", "Refund": 1, "Fines": 2500.0, "Make": "BMW", "Model": "X5", "Year": 2007},
    {"CarNumber": "C333CC177RUS", "Refund": 0, "Fines": 900.0, "Make": "Kia", "Model": "Rio", "Year": 2007},
    {"CarNumber": "D444DD197RUS", "Refund": 1, "Fines": 3200.0, "Make": "Audi", "Model": "A4", "Year": 2007},
    {"CarNumber": "E555EE750RUS", "Refund": 0, "Fines": 1500.0, "Make": "Volkswagen", "Model": "Golf", "Year": 2007}
])

fines = pd.concat([fines, temp], ignore_index=True)
fines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    object 
 1   Refund     930 non-null    int64  
 2   Fines      930 non-null    float64
 3   Make       930 non-null    object 
 4   Model      930 non-null    object 
 5   Year       930 non-null    int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 43.7+ KB


- удаляем 20 последних строк, добавляем 3 новых

In [38]:
owners = owners.iloc[:-20].reset_index(drop=True)

three_new = pd.DataFrame([
    {"CarNumber": "F666FF178RUS", "SURNAME": "Peldobaras"},
    {"CarNumber": "G777GG50RUS",  "SURNAME": "Chephuhov"},
    {"CarNumber": "H888HH97RUS",  "SURNAME": "Boriskov"}
])
owners = pd.concat([owners, three_new], ignore_index=True)
owners.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   CarNumber  514 non-null    object
 1   SURNAME    514 non-null    object
dtypes: object(2)
memory usage: 8.2+ KB


- Обьединяем правильно в 4х вариантах.
    1. Которые есть в каждом (логическое "И")
    2. Все номера из обоих (логическое "ИЛИ")
    3. Только из первого
    4. Только из второго

In [ ]:

df_inner = fines.merge(owners, on="CarNumber", how="inner")

df_outer = fines.merge(owners, on="CarNumber", how="outer")

df_left = fines.merge(owners, on="CarNumber", how="left")

df_right = fines.merge(owners, on="CarNumber", how="right")

len(df_inner)


- сводная таблица

In [40]:
fines.pivot_table(index="Make", columns="Year", values="Fines", aggfunc="sum")

Year,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
Make,,,,,,,,,,,,,,,,,,,,,
Audi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4200.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BMW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,6000.00,NaN,NaN,8594.59,NaN,NaN,6500.00,NaN
Ford,110294.59,408983.76,165883.76,64800.00,96989.17,162683.76,96589.17,125700.00,111789.17,184694.59,...,142678.35,103478.35,131500.00,139394.59,122678.35,209100.00,144289.17,263000.00,274089.17,78889.17
Kia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Skoda,2400.00,NaN,7300.00,11594.59,NaN,10294.59,600.00,5200.00,5200.00,91400.00,...,3100.00,500.00,500.00,12594.59,300.00,46394.59,300.00,8594.59,156200.00,9500.00
Toyota,12000.00,8594.59,2000.00,7200.00,NaN,NaN,NaN,14900.00,NaN,26400.00,...,24000.00,8594.59,38894.59,NaN,1000.00,NaN,900.00,9600.00,29194.59,18100.00
Volkswagen,30900.00,1600.00,NaN,11794.59,10300.00,34800.00,22400.00,57100.00,NaN,5800.00,...,9100.00,300.00,NaN,20800.00,1300.00,3400.00,2100.00,NaN,1000.00,NaN
Volvo,NaN,NaN,6800.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10200.00


- сохраняем в .csv

In [41]:
fines.to_csv("fines.csv", index=False)
owners.to_csv("owners.csv", index=False)
owners.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   CarNumber  514 non-null    object
 1   SURNAME    514 non-null    object
dtypes: object(2)
memory usage: 8.2+ KB


In [44]:
len(fines)

930